In [2]:
%pip install chromadb langchain-chroma langchain

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain-1.4.9-py3-none-any.whl.metadata (6.2 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached pydantic_settings-2.16.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached pybase64-1.5.1-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.31.1-cp314-cp314-win_amd64.whl.metadata (6.1 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import chromadb
import csv
import re
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
DB_PATH = r"D:\LLM TASKS\LLM_Vector_Embeddings\chroma_db"

client = chromadb.PersistentClient(path=DB_PATH)

collection = client.get_collection(
    name="ai_education_documents"
)

print("ChromaDB Connected Successfully!")
print("Total Documents:", collection.count())

ChromaDB Connected Successfully!
Total Documents: 30


In [3]:
DATASET_DIR = Path("dataset")
DOCUMENTS_DIR = DATASET_DIR / "documents"

DATASET_DIR.mkdir(exist_ok=True)
DOCUMENTS_DIR.mkdir(exist_ok=True)

print("Dataset Folder:", DATASET_DIR)
print("Documents Folder:", DOCUMENTS_DIR)
print("TXT Files:", len(list(DOCUMENTS_DIR.glob("*.txt"))))

Dataset Folder: dataset
Documents Folder: dataset\documents
TXT Files: 0


In [4]:
embedding_function = DefaultEmbeddingFunction()

class ChromaEmbeddings(Embeddings):

    def embed_documents(self, texts):
        return embedding_function(texts)

    def embed_query(self, text):
        return embedding_function([text])[0]

print("Embedding Function Ready!")

Embedding Function Ready!


In [5]:
vector_store = Chroma(
    client=client,
    collection_name="ai_education_documents",
    embedding_function=ChromaEmbeddings()
)

print("LangChain Integrated with ChromaDB!")

LangChain Integrated with ChromaDB!


In [6]:
queries = [
    "How does artificial intelligence help students learn?",
    "How can computers understand human language?",
    "How can teachers identify weak students?",
    "How is student information protected?",
    "How do online classes help students?",
    "How can lessons be personalized?",
    "What are the uses of AI chatbots?",
    "How can AI bias be reduced?",
    "How does cloud computing help education?",
    "How can online learning systems be secured?",
    "How does machine learning improve education?",
    "How can student attendance be analyzed?",
    "How does automatic translation work?",
    "How can technology support teachers?",
    "How can students improve their learning performance?"
]

print("Total Queries:", len(queries))

Total Queries: 15


In [7]:
data = collection.get(
    include=["documents", "metadatas"]
)

all_documents = []

for i, document in enumerate(data["documents"]):

    all_documents.append({
        "document": document,
        "metadata": data["metadatas"][i] or {}
    })

print("Total Documents Retrieved:", len(all_documents))

Total Documents Retrieved: 30


In [8]:
semantic_results = []
similarity_scores = []
keyword_results = []

print("Result Lists Initialized!")

Result Lists Initialized!


In [9]:
for query_number, query in enumerate(queries, start=1):

    results = vector_store.similarity_search_with_score(
        query,
        k=min(5, collection.count())
    )

    for rank, (doc, distance) in enumerate(results, start=1):

        metadata = doc.metadata
        similarity = 1 / (1 + float(distance))

        semantic_results.append([
            query_number,
            query,
            rank,
            metadata.get("title", "N/A"),
            metadata.get("category", "N/A"),
            metadata.get("source", "N/A"),
            round(float(distance), 4),
            round(similarity, 4),
            doc.page_content
        ])

        similarity_scores.append([
            query_number,
            query,
            rank,
            round(float(distance), 4),
            round(similarity, 4)
        ])

print("Semantic Search Completed!")
print("Total Semantic Results:", len(semantic_results))

Semantic Search Completed!
Total Semantic Results: 75


In [10]:
for query_number, query in enumerate(queries, start=1):

    query_words = set(
        re.findall(r"[a-zA-Z0-9]+", query.lower())
    )

    matched_documents = []

    for item in all_documents:

        text = (
            item["document"] + " " +
            str(item["metadata"].get("title", "")) + " " +
            str(item["metadata"].get("category", ""))
        ).lower()

        document_words = set(
            re.findall(r"[a-zA-Z0-9]+", text)
        )

        matches = query_words.intersection(document_words)

        score = len(matches) / max(len(query_words), 1)

        if score > 0:
            matched_documents.append((score, item))

    matched_documents.sort(
        key=lambda x: x[0],
        reverse=True
    )

    for rank, (score, item) in enumerate(
        matched_documents[:5], start=1
    ):

        metadata = item["metadata"]

        keyword_results.append([
            query_number,
            query,
            rank,
            metadata.get("title", "N/A"),
            metadata.get("category", "N/A"),
            round(score, 4),
            item["document"]
        ])

print("Keyword Search Completed!")
print("Total Keyword Results:", len(keyword_results))

Keyword Search Completed!
Total Keyword Results: 74


In [12]:
def save_csv(filename, headers, rows):

    file_path = DATASET_DIR / filename

    with open(
        file_path,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as file:

        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)

    print("Created:", file_path)

In [13]:
save_csv(
    "search_results.csv",
    [
        "Query_Number", "Query", "Rank", "Title",
        "Category", "Source", "Distance",
        "Similarity_Score", "Document"
    ],
    semantic_results
)

Created: dataset\search_results.csv


In [14]:
save_csv(
    "similarity_scores.csv",
    [
        "Query_Number", "Query", "Rank",
        "Distance", "Similarity_Score"
    ],
    similarity_scores
)

Created: dataset\similarity_scores.csv


In [15]:
save_csv(
    "keyword_results.csv",
    [
        "Query_Number", "Query", "Rank",
        "Title", "Category",
        "Keyword_Match_Score", "Document"
    ],
    keyword_results
)

Created: dataset\keyword_results.csv


In [16]:
save_csv(
    "queries.csv",
    ["Query_Number", "Query"],
    [
        [i, query]
        for i, query in enumerate(queries, start=1)
    ]
)

Created: dataset\queries.csv


In [17]:
print("======================================")
print("SEMANTIC DOCUMENT SEARCH COMPLETED")
print("======================================")

print("Total Queries:", len(queries))
print("Total Documents:", collection.count())
print("Semantic Results:", len(semantic_results))
print("Similarity Scores:", len(similarity_scores))
print("Keyword Results:", len(keyword_results))

print("\nAll CSV files saved successfully!")

SEMANTIC DOCUMENT SEARCH COMPLETED
Total Queries: 15
Total Documents: 30
Semantic Results: 75
Similarity Scores: 75
Keyword Results: 74

All CSV files saved successfully!


In [18]:
user_query = input("Enter your search query: ")

results = vector_store.similarity_search_with_score(
    user_query,
    k=5
)

print("\nTop 5 Search Results:\n")

for rank, (doc, distance) in enumerate(results, start=1):

    print("Rank:", rank)
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Distance:", round(float(distance), 4))
    print("Document:", doc.page_content)
    print("-" * 60)


Top 5 Search Results:

Rank: 1
Title: Artificial Intelligence in Education
Distance: 0.3731
Document: Artificial intelligence helps students learn through smart learning platforms, automated assessments, and personalized recommendations. AI tools support teachers by analyzing student performance and providing useful learning suggestions.
------------------------------------------------------------
Rank: 2
Title: Document 10
Distance: 0.4523
Document: Responsible Artificial Intelligence in education focuses on fairness, privacy, transparency, and accountability. AI systems should protect student information and avoid biased decisions while supporting effective learning.
------------------------------------------------------------
Rank: 3
Title: Document 1
Distance: 0.4537
Document: Artificial Intelligence is transforming education by providing personalized learning experiences. AI systems analyze student performance and recommend learning materials based on individual needs. It helps t